# 📏 Model Değerlendirme (Evaluation Metrics & Cross Validation)

Modelin performansını güvenilir bir şekilde ölçmek (K-Fold Cross Validation) ve sonuçları görselleştirmek (Confusion Matrix, ROC Curve) için kullanılan şablonlar.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import KFold, StratifiedKFold, cross_val_score
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, roc_auc_score, roc_curve

## 1. Çapraz Doğrulama (Cross Validation)
Veriyi K parçaya (Fold) böler. Modeli K defa eğitip test eder. Sonuçların ortalamasını alarak daha gerçekçi bir skor üretir.

In [ ]:
def evaluate_with_cv(model, X, y, task='regression', n_splits=5):
    """
    Regresyon için RMSE, Sınıflandırma için Accuracy hesaplar.
    """
    if task == 'regression':
        # Regresyon verilerinde standart KFold yeterlidir.
        kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)
        scores = cross_val_score(model, X, y, cv=kf, scoring='neg_root_mean_squared_error')
        # Negatif dönen skoru pozitife çeviriyoruz
        rmse_scores = -scores
        print(f"{n_splits}-Fold CV RMSE Skorları: {rmse_scores}")
        print(f"Ortalama RMSE: {rmse_scores.mean():.4f} (+/- {rmse_scores.std():.4f})")
        
    else:
        # Sınıflandırmada sınıfların (0-1) her fold'da eşit dağılması için StratifiedKFold kullanılır.
        skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
        scores = cross_val_score(model, X, y, cv=skf, scoring='accuracy')
        print(f"{n_splits}-Fold CV Accuracy Skorları: {scores}")
        print(f"Ortalama Accuracy: {scores.mean():.4f} (+/- {scores.std():.4f})")

## 2. Regresyon Hata Raporu

In [ ]:
def plot_regression_metrics(y_true, y_pred):
    """
    Tahmin ve Gerçek değerlerin sayısal hata metriklerini hesaplar ve saçılam (scatter) grafiğini çizer.
    """
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)
    
    print("--- Regresyon Sonuçları ---")
    print(f"RMSE (Kök Ortalama Kare Hata) : {rmse:.4f}")
    print(f"MAE (Ortalama Mutlak Hata)    : {mae:.4f}")
    print(f"R2 Skoru (Açıklanabilirlik)   : {r2:.4f}")
    
    plt.figure(figsize=(8, 8))
    sns.scatterplot(x=y_true, y=y_pred, alpha=0.6, color="blue")
    
    # Mükemmel tahmin çizgisi (y = x)
    min_val = min(y_true.min(), y_pred.min())
    max_val = max(y_true.max(), y_pred.max())
    plt.plot([min_val, max_val], [min_val, max_val], color="red", linestyle="--")
    
    plt.xlabel("Gerçek Değerler")
    plt.ylabel("Tahmin Edilen Değerler")
    plt.title("Gerçek vs Tahmin Grafiği")
    plt.show()

## 3. Sınıflandırma Hata Raporu (Karmaşıklık Matrisi)

In [ ]:
def plot_classification_metrics(y_true, y_pred, y_prob=None):
    """
    Doğruluk (Accuracy) ve F1 Skorlarını gösterir, Confusion Matrix (Karmaşıklık Matrisi) çizer.
    y_prob (Olasılık tahmini) verilirse ROC-AUC skoru da hesaplar.
    """
    print("--- Sınıflandırma Raporu ---")
    print(classification_report(y_true, y_pred))
    
    if y_prob is not None:
        auc = roc_auc_score(y_true, y_prob)
        print(f"ROC-AUC Skoru: {auc:.4f}")
    
    cm = confusion_matrix(y_true, y_pred)
    
    plt.figure(figsize=(6, 5))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues")
    plt.xlabel("Tahmin Edilen Sınıf")
    plt.ylabel("Gerçek Sınıf")
    plt.title("Confusion Matrix")
    plt.show()